In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

RANDOM_SEED   = 42
LEARNING_RATE = 0.001
BATCH_SIZE    = 256
EPOCHS_CNN    = 30
EPOCHS_AE     = 30
device        = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

COL_NAMES = [
    'duration', 'protocol_type', 'service', 'flag',
    'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot',
    'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
    'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
    'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate',
    'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate',
    'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate',
    'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate',
    'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate',
    'dst_host_srv_serror_rate', 'dst_host_rerror_rate',
    'dst_host_srv_rerror_rate', 'label', 'difficulty'
]

SELECTED_FEATURES = [
    'src_bytes', 'service', 'dst_bytes', 'flag',
    'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate',
]

def find_file(filename):
    search_paths = [
        (Path.home() / ".cache" / "kagglehub" / "datasets" / "hassan06" / "nslkdd" / "versions" / "1"),
        Path("."),
        Path.home() / "Downloads",
    ]
    for base in search_paths:
        candidate = base / filename
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"No se encontro '{filename}'.")

df = pd.read_csv(find_file('KDDTrain+_20Percent.txt'), header=None, names=COL_NAMES).drop(columns=['difficulty'])
print(f"Instancias: {len(df):,}")
print(df['label'].value_counts())

Instancias: 25,192
label
normal             13449
neptune             8282
ipsweep              710
satan                691
portsweep            587
smurf                529
nmap                 301
back                 196
teardrop             188
warezclient          181
pod                   38
guess_passwd          10
warezmaster            7
buffer_overflow        6
imap                   5
rootkit                4
multihop               2
phf                    2
ftp_write              1
land                   1
loadmodule             1
spy                    1
Name: count, dtype: int64


In [2]:
protocol_map = {'tcp': 1, 'udp': 2, 'icmp': 3}
df['protocol_type'] = df['protocol_type'].str.lower().map(protocol_map).fillna(0)
df['service'] = LabelEncoder().fit_transform(df['service'].astype(str))
df['flag']    = LabelEncoder().fit_transform(df['flag'].astype(str))

DOS   = {'back','land','neptune','pod','smurf','teardrop','apache2','udpstorm','processtable','mailbomb'}
PROBE = {'ipsweep','nmap','portsweep','satan','mscan','saint'}
R2L   = {'ftp_write','guess_passwd','imap','multihop','phf','spy','warezclient','warezmaster',
         'sendmail','named','snmpgetattack','snmpguess','xlock','xsnoop','worm'}
U2R   = {'buffer_overflow','loadmodule','perl','rootkit','httptunnel','ps','sqlattack','xterm'}

def map_label(label):
    label = label.lower().strip()
    if label == 'normal': return 1
    if label in DOS:      return 2
    if label in PROBE:    return 3
    if label in R2L:      return 4
    if label in U2R:      return 5
    return 2

df['label'] = df['label'].apply(map_label)

SELECTED_FEATURES = [
    'src_bytes', 'service', 'dst_bytes', 'flag',
    'diff_srv_rate', 'same_srv_rate', 'dst_host_srv_count', 'dst_host_same_srv_rate',
]

X = df[SELECTED_FEATURES].values.astype(np.float32)
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# CORREGIDO: MinMaxScaler ajustado solo sobre train (post-split), sin data leakage
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

CLASS_NAMES  = {1: 'Normal', 2: 'DoS', 3: 'Probe', 4: 'R2L', 5: 'U2R'}
y_train_bin  = (y_train != 1).astype(int)
y_test_bin   = (y_test  != 1).astype(int)

X_train_cnn    = X_train
y_train_multi  = y_train
X_train_autoenc = X_train[y_train == 1]  # Normal = clase 1 en NSL-KDD

N_CLASSES = len(np.unique(y_train_multi))
INPUT_DIM = X_train_cnn.shape[1]

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"AE Train (Normal): {len(X_train_autoenc):,}")
print(f"INPUT_DIM: {INPUT_DIM} | N_CLASSES: {N_CLASSES}")

Train: 20,153 | Test: 5,039
AE Train (Normal): 10,759
INPUT_DIM: 8 | N_CLASSES: 5


In [3]:
class DPACK(nn.Module):
    def __init__(self, input_dim, n_classes):
        super(DPACK, self).__init__()
        self.input_dim = input_dim
        self.conv1 = nn.Sequential(nn.Conv1d(1, 32, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(32))
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.conv2 = nn.Sequential(nn.Conv1d(32, 64, kernel_size=6, stride=1, padding=5), nn.ReLU(), nn.BatchNorm1d(64))
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)
        self._conv_out_size = self._get_conv_output_size()
        self.fc5 = nn.Sequential(nn.Linear(self._conv_out_size, 1024), nn.ReLU(), nn.BatchNorm1d(1024))
        self.fc6 = nn.Sequential(nn.Linear(1024, 25), nn.ReLU(), nn.BatchNorm1d(25))
        self.fc7 = nn.Linear(25, n_classes)
        self.ae_fc8  = nn.Sequential(nn.Linear(1024, 512), nn.ReLU())
        self.ae_fc9  = nn.Sequential(nn.Linear(512,  256), nn.ReLU())
        self.ae_fc10 = nn.Sequential(nn.Linear(256,  512), nn.ReLU())
        self.ae_fc11 = nn.Sequential(nn.Linear(512, 1024), nn.ReLU())
        self.ae_out  = nn.Linear(1024, input_dim)

    def _get_conv_output_size(self):
        with torch.no_grad():
            dummy = torch.zeros(1, 1, self.input_dim)
            out   = self.pool2(self.conv2(self.pool1(self.conv1(dummy))))
            return int(out.numel())

    def forward(self, x):
        h = x.unsqueeze(1)
        h = self.pool1(self.conv1(h))
        h = self.pool2(self.conv2(h))
        h = h.view(h.size(0), -1)
        h5      = self.fc5(h)
        cnn_out = self.fc7(self.fc6(h5))
        ae_out  = self.ae_out(self.ae_fc11(self.ae_fc10(self.ae_fc9(self.ae_fc8(h5)))))
        return cnn_out, ae_out, h5


def train_cnn_phase(model, X_tr, y_tr, epochs, batch_size, lr, device):
    model.train()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_tr), torch.LongTensor(y_tr - 1)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_loss, correct, total = 0.0, 0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            cnn_out, ae_out, _ = model(X_batch)
            loss = criterion(cnn_out, y_batch) + nn.MSELoss()(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (cnn_out.argmax(dim=1) == y_batch).sum().item()
            total      += y_batch.size(0)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoca {epoch+1:>3}/{epochs} | Loss: {total_loss/len(loader):.4f} | Acc: {100*correct/total:.2f}%")
    return model


def train_autoencoder_phase(model, X_ae, epochs, batch_size, lr, device):
    model.train()
    ae_params = (list(model.fc5.parameters())    + list(model.ae_fc8.parameters()) +
                 list(model.ae_fc9.parameters()) + list(model.ae_fc10.parameters()) +
                 list(model.ae_fc11.parameters()) + list(model.ae_out.parameters()))
    optimizer = optim.Adam(ae_params, lr=lr)
    criterion = nn.MSELoss()
    loader    = DataLoader(TensorDataset(torch.FloatTensor(X_ae)),
                           batch_size=batch_size, shuffle=True, num_workers=0)
    for epoch in range(epochs):
        total_mse = 0.0
        for (X_batch,) in loader:
            X_batch = X_batch.to(device)
            optimizer.zero_grad()
            _, ae_out, _ = model(X_batch)
            loss = criterion(ae_out, X_batch)
            loss.backward()
            optimizer.step()
            total_mse += loss.item()
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoca {epoch+1:>3}/{epochs} | MSE: {total_mse/len(loader):.6f}")
    return model


def compute_threshold(model, X_benign, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X_benign), batch_size):
            X_batch = torch.FloatTensor(X_benign[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    mse_arr = np.array(mse_list)
    max_mse, p99, std = np.max(mse_arr), np.percentile(mse_arr, 99), np.std(mse_arr)
    threshold = p99 if (max_mse - p99) > 3 * std else max_mse
    print(f"  Umbral = {threshold:.6f} ({'percentil 99' if threshold == p99 else 'max MSE'})")
    return threshold


torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model = DPACK(input_dim=INPUT_DIM, n_classes=N_CLASSES).to(device)
print(f"Parametros totales: {sum(p.numel() for p in model.parameters()):,}")

print("\nFASE 1 - CNN")
model = train_cnn_phase(model, X_train_cnn, y_train_multi, EPOCHS_CNN, BATCH_SIZE, LEARNING_RATE, device)

print("\nFASE 2 - Autoencoder")
model = train_autoencoder_phase(model, X_train_autoenc, EPOCHS_AE, BATCH_SIZE, LEARNING_RATE, device)

print("\nUMBRAL")
threshold = compute_threshold(model, X_train_autoenc, device)

Parametros totales: 1,690,549

FASE 1 - CNN


  Epoca   1/30 | Loss: 0.5279 | Acc: 90.13%


  Epoca   5/30 | Loss: 0.0990 | Acc: 97.25%


  Epoca  10/30 | Loss: 0.0773 | Acc: 97.65%


  Epoca  15/30 | Loss: 0.0671 | Acc: 97.88%


...

  Epoca  15/30 | MSE: 0.000783


  Epoca  20/30 | MSE: 0.000155


  Epoca  25/30 | MSE: 0.000189


  Epoca  30/30 | MSE: 0.000498

UMBRAL


  Umbral = 0.001506 (percentil 99)


In [4]:
def predict(model, X, threshold, device, batch_size=512):
    model.eval()
    mse_list = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.FloatTensor(X[i:i+batch_size]).to(device)
            _, ae_out, _ = model(X_batch)
            mse_list.extend(((ae_out - X_batch) ** 2).mean(dim=1).cpu().numpy())
    return (np.array(mse_list) > threshold).astype(int)


def compute_metrics(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    accuracy  = (tp + tn) / (tp + tn + fp + fn)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    far       = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr       = fn / (tp + fn) if (tp + fn) > 0 else 0.0
    return {'Accuracy': accuracy, 'Precision': precision, 'Recall': recall,
            'F1': f1, 'FAR': far, 'FNR': fnr, 'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn}


y_pred  = predict(model, X_test, threshold, device)
metrics = compute_metrics(y_test_bin, y_pred)

SEP = '=' * 50
print("D-PACK - Dataset NSL-KDD")
print(SEP)
for k in ('Accuracy', 'Precision', 'Recall', 'F1', 'FAR', 'FNR'):
    print(f"  {k:<12}: {metrics[k]*100:.2f}%")
print(SEP)
print(f"  TP={int(metrics['TP']):,}  TN={int(metrics['TN']):,}  FP={int(metrics['FP']):,}  FN={int(metrics['FN']):,}")

print("\nRecall por clase:")
for cls_id, cls_name in CLASS_NAMES.items():
    mask         = (y_test == cls_id)
    expected_bin = 0 if cls_id == 1 else 1
    rec = ((y_pred == expected_bin) & mask).sum() / mask.sum() if mask.sum() > 0 else 0.0
    print(f"  {cls_name:<8}: {rec*100:.1f}%  ({mask.sum():,} instancias)")

D-PACK - Dataset NSL-KDD
  Accuracy    : 91.25%
  Precision   : 98.04%
  Recall      : 82.89%
  F1          : 89.83%
  FAR         : 1.45%
  FNR         : 17.11%
  TP=1,947  TN=2,651  FP=39  FN=402

Recall por clase:
  Normal  : 98.6%  (2,690 instancias)
  DoS     : 91.7%  (1,847 instancias)
  Probe   : 54.6%  (458 instancias)
  R2L     : 9.5%  (42 instancias)
  U2R     : 0.0%  (2 instancias)
